# 第 6 章 · 底层控制：电流环 → 速度环 → 位置环，再到阻抗与全身控制

配套讲义：仓库里 `robot-system-architecture/notes/lessons/06-low-level-control.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（PD 增益、电流、角度是编的；刚度用了动画里的 300 N/m 和编的 2000 N/m）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 PD + 前馈 | $K_p(q^*-q) + K_d(\dot q^*-\dot q) + \tau_{ff}$ | `pd_ff(...)` | 5 − 1 + 3 = 7 N·m |
| 第 2 步 FOC | Clarke → Park | `clarke`、`park` | 三相电流 → $i_d = 0$、$i_q = 5$ A |
| 第 2 步 电流即力矩 | $\tau = K_t i_q$ | `0.1 * i_q` | 0.5 N·m |
| 第 4 步 阻抗撞墙 | 稳态 $F \approx K \times$ 深度 | `impedance_wall(k)` | K = 300 → 5.98 N；K = 2000 → 39.2 N |
| 公式表 临界阻尼 | $D = 2\sqrt{Km}$ | `critical_damping` | 34.64、89.44 N·s/m |

PD 增益、FOC 的电流和角度、阻抗仿真里的质量都是编的；刚度只用了动画里的 300 N/m 和另一个编的 2000 N/m。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `pd_ff`**：关节 PD + 前馈：τ = K_p (q* − q) + K_d (q̇* − q̇) + τ_ff。

In [ ]:
def pd_ff(q_ref: float, q: float, dq_ref: float, dq: float, kp: float, kd: float, tau_ff: float) -> float:
    """关节 PD + 前馈：τ = K_p (q* − q) + K_d (q̇* − q̇) + τ_ff。"""
    return kp * (q_ref - q) + kd * (dq_ref - dq) + tau_ff

**函数 `clarke`**：Clarke 变换：三相 (a, b, c) → 两轴 (α, β)，幅值不变的写法。

In [ ]:
def clarke(ia: float, ib: float, ic: float) -> tuple[float, float]:
    """Clarke 变换：三相 (a, b, c) → 两轴 (α, β)，幅值不变的写法。"""
    i_alpha = (2 / 3) * (ia - 0.5 * ib - 0.5 * ic)
    i_beta = (2 / 3) * (np.sqrt(3) / 2) * (ib - ic)
    return i_alpha, i_beta

**函数 `park`**：Park 变换：按转子电角度 θ_e 旋转到 (d, q)；就是一个 2×2 旋转矩阵。

In [ ]:
def park(i_alpha: float, i_beta: float, theta_e: float) -> tuple[float, float]:
    """Park 变换：按转子电角度 θ_e 旋转到 (d, q)；就是一个 2×2 旋转矩阵。"""
    c, s = np.cos(theta_e), np.sin(theta_e)
    return c * i_alpha + s * i_beta, -s * i_alpha + c * i_beta

**函数 `three_phase`**：反过来：给定 i_d、i_q 和角度，合成三相电流（反 Park + 反 Clarke）。

In [ ]:
def three_phase(i_d: float, i_q: float, theta_e: float) -> tuple[float, float, float]:
    """反过来：给定 i_d、i_q 和角度，合成三相电流（反 Park + 反 Clarke）。"""
    phases = (0.0, -2 * np.pi / 3, 2 * np.pi / 3)
    return tuple(i_d * np.cos(theta_e + p) - i_q * np.sin(theta_e + p) for p in phases)  # type: ignore[return-value]

**函数 `critical_damping`**：临界阻尼 D = 2 √(K m)。

In [ ]:
def critical_damping(k: float, m: float) -> float:
    """临界阻尼 D = 2 √(K m)。"""
    return 2 * np.sqrt(k * m)

**函数 `impedance_wall`**：一维阻抗控制撞墙：墙在 x = 0（刚度 1e5 N/m 的硬弹簧），目标在墙后 depth。返回最后的接触力（N）。

In [ ]:
def impedance_wall(k: float, m: float = 1.0, depth: float = 0.02, t_end: float = 2.0, dt: float = 1e-4) -> float:
    """一维阻抗控制撞墙：墙在 x = 0（刚度 1e5 N/m 的硬弹簧），目标在墙后 depth。返回最后的接触力（N）。"""
    d = critical_damping(k, m)
    k_wall = 1e5
    x, v = -0.10, 0.0  # 从墙前 10 cm 出发
    for _ in range(int(t_end / dt)):
        f_ctrl = k * (depth - x) + d * (0.0 - v)  # F = K (x* − x) + D (ẋ* − ẋ)
        f_wall = -k_wall * x if x > 0 else 0.0
        v += (f_ctrl + f_wall) / m * dt
        x += v * dt
    return k_wall * x if x > 0 else 0.0

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：PD + 前馈（编的：q* = 1.0，q = 0.9，q̇ = 0.5，Kp = 50，Kd = 2，τ_ff = 3）")
tau = pd_ff(1.0, 0.9, 0.0, 0.5, kp=50, kd=2, tau_ff=3)
print(f"  τ = 50 × 0.1 + 2 × (0 − 0.5) + 3 = {tau:.1f} N·m")

In [ ]:
print("\n第二步：FOC——三相电流经过 Clarke、Park 变成两个直流量")
theta_e = np.radians(40.0)
ia, ib, ic = three_phase(0.0, 5.0, theta_e)
print(f"  三相 a, b, c = {ia:+.3f}, {ib:+.3f}, {ic:+.3f} A（加起来 {ia + ib + ic:+.1e}）")
i_al, i_be = clarke(ia, ib, ic)
i_d, i_q = park(i_al, i_be, theta_e)
print(f"  Clarke → α, β = {i_al:+.3f}, {i_be:+.3f}；Park → d, q = {i_d:+.3f}, {i_q:+.3f}")
print(f"  τ = K_t i_q：K_t = 0.1 N·m/A → τ = {0.1 * i_q:.2f} N·m")

In [ ]:
print("\n第四步：阻抗控制撞墙，目标在墙后 2 cm，稳态力 ≈ K × 0.02")
for k in (300.0, 2000.0):
    f = impedance_wall(k)
    print(f"  K = {k:>6.0f} N/m：仿真稳态力 {f:6.2f} N，估算 K × 0.02 = {k * 0.02:.1f} N")

In [ ]:
print("\n公式表：临界阻尼 D = 2 √(K m)，m = 1 kg")
for k in (300.0, 2000.0):
    print(f"  K = {k:.0f} → D = {critical_damping(k, 1.0):.2f} N·s/m")

## 3. 代码里才看得到的

- 仿真稳态力比 K × 2 cm 略小（5.98 而不是 6.0）：墙本身也是弹簧（1e5 N/m），手会陷进去一点，离目标的距离就不到 2 cm 了。测试放宽到 3%。
- 三相电流加起来是 0（打印成 −8.9e-16 是浮点误差）；所以 Clarke 变换从 3 个数变成 2 个数不丢信息。
- Park 变换就是一个 2×2 旋转矩阵：按转子角度把 (α, β) 转到跟着转子走的 (d, q) 上，交流量就变成了直流量。

## 4. 自己改一改

- 把 `impedance_wall` 的阻尼改成临界阻尼的十分之一，看撞墙后力会不会来回弹。
- 把 FOC 例子里的 `i_d` 改成 −2（弱磁），看三相电流和力矩怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 内环快、外环慢，一层追一层；电流就是力矩。
> 位置控制硬顶，阻抗控制像弹簧；全身控制是每毫秒解一道带约束的题。